### Model Parameter 설정

In [2]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 값은 출력하지 않고 설정 여부만 확인 (사용할 제공자의 키만 있으면 됨)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    print(f"{name}: {'설정됨' if key else '없음'}")

GROQ_API_KEY: 설정됨
OPENAI_API_KEY: 설정됨


#### LLM 설정 (Groq / OpenAI 선택)

- **Groq**와 **OpenAI** 두 가지 모델을 모두 사용할 수 있습니다. (Groq는 OpenAI 호환 API라서 `ChatOpenAI`로 연결)
- 아래 셀의 `PROVIDER` 값(`"groq"` 또는 `"openai"`)만 바꾸면 이후 모든 예제의 모델이 바뀝니다. 값을 바꾼 뒤에는 이 셀부터 다시 실행하세요.
- `get_llm(temperature=..., max_tokens=..., top_p=...)`처럼 **모델 파라미터는 `get_llm()`에 그대로 전달**합니다.
- `get_llm(light=True)`는 같은 질문을 여러 번 실행하는 실험용 **경량 모델**을 만듭니다. (Groq: `gpt-oss-20b`, OpenAI: `gpt-4o-mini`)
- `reasoning_effort`는 Groq `gpt-oss` 추론 모델 전용 파라미터라서, OpenAI를 선택하면 무시됩니다.
- 사용할 제공자의 키(`GROQ_API_KEY` 또는 `OPENAI_API_KEY`)가 `.env`에 있어야 합니다. OpenAI를 선택하면 사용 요금이 발생합니다.

In [3]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
        "light_model": "openai/gpt-oss-20b",  # 반복 실행용 경량 모델 (비용/한도 부담이 적음)
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-5.6-luna",
        "light_model": "gpt-4o-mini",
    },
}
MAX_RETRIES = 5  # 429(분당 한도), 503 등 일시 오류 자동 재시도 횟수


def get_llm(temperature: float | None = None, provider: str | None = None, light: bool = False,
            reasoning_effort: str | None = None, verbose: bool = True, **params) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다.
    temperature: None 이면 모델 기본값 사용 / light: 경량 모델 사용 / reasoning_effort: Groq gpt-oss 전용
    params: ChatOpenAI 에 그대로 전달할 모델 파라미터 (max_tokens, top_p, presence_penalty, frequency_penalty 등)
    verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    if reasoning_effort and provider == "groq":
        params["reasoning_effort"] = reasoning_effort  # gpt-oss(Groq) 전용 → OpenAI gpt-4o-mini 는 지원하지 않아 전달하지 않음
        
    llm = ChatOpenAI(
        api_key=config["api_key"],
        base_url=config["base_url"],
        model=config["light_model"] if light else config["model"],
        temperature=temperature,
        max_retries=MAX_RETRIES,
        **params,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name} | temperature={temperature}")
    return llm

### 1) 모델 파라미터 설정
#### Model Parameter
* Temperature (온도): 이 파라미터는 모델의 예측 불확실성을 조절합니다. 
    * 낮은 온도는 모델이 가장 확률이 높은 다음 단어를 선택하게 하여 안정적인 결과를 내고, 높은 온도는 확률이 낮은 단어까지 선택지에 포함시켜 더 창의적이고 예측 불가능한 답변을 생성합니다. 
    * 0.2 (낮은 값): 더 예측 가능하고 일반적인 이야기
    * 1.2 (높은 값): 새로운 요소가 등장하며, 더 창의적이고 흥미로운 이야기 생성

* Presence Penalty & Frequency Penalty: 두 파라미터 모두 토큰(단어)의 반복성을 제어합니다.
    * Presence Penalty는 이전에 나온 단어나 주제에 '페널티'를 주어 새로운 단어를 사용하게 합니다. 값이 높을수록 새로운 내용을 더 자주 시도합니다.
    * Frequency Penalty는 단어가 등장한 빈도에 비례하여 '페널티'를 줍니다. 값이 높을수록 특정 단어를 반복적으로 사용하는 것을 강하게 억제하여 더 다양한 어휘를 사용하게 합니다. 
    * 0.0 (낮은 값): 기존에 자주 등장하는 단어와 구조 유지
    * 1.5 (높은 값): 새로운 표현과 독창적인 아이디어 등장
* Top_p: 이 파라미터는 다음 토큰을 선택할 때 고려할 단어의 범위를 조절합니다. Top_p가 0.3이면 확률이 높은 상위 30%의 단어 중에서만 선택하고, 0.95면 확률이 낮은 단어들까지 포함하여 거의 모든 단어를 고려합니다. Top_p가 높을수록 모델의 선택지가 넓어져 더 자유로운 응답이 가능합니다.    

In [4]:
# 제공자/모델은 위 공통 설정 셀의 get_llm 을 사용합니다.
# 이 셀의 두 모델은 provider="openai" 로 직접 지정해 OpenAI 기본 모델(LLM_CONFIGS 의 "model")을 사용합니다. (PROVIDER 기본값은 Groq)

# 보수적인 설정 (일관되고 예측 가능한 답변)
llm_conservative = get_llm(
    reasoning_effort="low",  # Groq gpt-oss 는 추론 토큰도 max_tokens 에 포함 → 추론량 최소화 (OpenAI 선택 시 무시)
    temperature=0.1,  # 매우 낮은 온도 - 예측 가능하고 일관된 출력
    presence_penalty=-0.5,  # 기존 패턴과 표현을 강하게 선호
    frequency_penalty=-0.3,  # 반복적 표현 허용, 일관성 증대
    max_tokens=350,  # 두 설정의 응답 길이 상한을 같게 맞춤 (길이가 아닌 다른 파라미터의 효과를 비교하기 위함)
    top_p=0.3,  # 매우 제한적인 토큰 선택 - 안전하고 예측 가능한 단어만
    provider="openai"
)

# 창의적인 설정 (독창적이고 예측 불가능한 답변)
llm_creative = get_llm(
    reasoning_effort="low",  # Groq gpt-oss 는 추론 토큰도 max_tokens 에 포함 → 추론량 최소화 (OpenAI 선택 시 무시)
    temperature=1.5,  # 매우 높은 온도 - 창의적이고 예측 불가능한 출력
    presence_penalty=1.5,  # 새로운 주제와 개념을 강하게 유도
    frequency_penalty=1.2,  # 반복을 강하게 억제하여 매우 다양한 표현 생성
    max_tokens=350,  # 보수적 설정과 동일한 상한
    top_p=0.95,  # 거의 모든 토큰 후보에서 선택 - 최대 창의성
    provider="openai"
)

# 질문 설정: 답이 하나로 정해지지 않는 짧은 질문 (이름 하나)
# 한 번의 응답은 우연일 수 있으므로, 같은 질문을 여러 번 실행해 응답이 얼마나 달라지는지 비교합니다.
# (temperature 등의 효과는 응답 하나가 아니라 반복 실행했을 때의 분포에서 드러납니다.)
question = """
반려 강아지 이름을 하나만 지어 주세요. 이름만 출력하세요.
"""
N_RUNS = 5  # 설정별 반복 실행 횟수

experiments = [
    ("CONSERVATIVE 설정 (온도:0.1, presence:-0.5, frequency:-0.3, top_p:0.3)", "같은 이름이 반복될 것으로 기대", llm_conservative),
    ("CREATIVE 설정 (온도:1.5, presence:1.5, frequency:1.2, top_p:0.95)", "실행할 때마다 다른 이름이 나올 것으로 기대", llm_creative),
]

for title, expectation, llm in experiments:
    print("=" * 60)
    print(f" {title}")
    print(f"  → {expectation}")
    print("=" * 60)

    # 같은 질문을 N_RUNS 번 실행 (동시 실행 수는 3으로 제한)
    answers = [r.content.strip() for r in llm.batch([question] * N_RUNS, config={"max_concurrency": 3})]
    print(f"서로 다른 응답 수: {len(set(answers))} / {N_RUNS}")
    for i, answer in enumerate(answers, 1):
        print(f"  {i}. {answer}")
    print()

print("\n" + "=" * 60)
print(" 파라미터 차이점 요약:")
print("   • Temperature: 0.1 vs 1.5 (15배 차이)")
print("   • Presence Penalty: -0.5 vs 1.5 (기존 패턴 선호 vs 새로운 개념 강요)")
print("   • Frequency Penalty: -0.3 vs 1.2 (반복 허용 vs 강한 다양성)")
print("   • Top-p: 0.3 vs 0.95 (제한적 vs 거의 무제한 토큰 선택)")
print("=" * 60)

[LLM 생성] provider=openai | model=gpt-5.6-luna | temperature=0.1
[LLM 생성] provider=openai | model=gpt-5.6-luna | temperature=1.5
 CONSERVATIVE 설정 (온도:0.1, presence:-0.5, frequency:-0.3, top_p:0.3)
  → 같은 이름이 반복될 것으로 기대


OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': "Unsupported parameter: 'top_p' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'top_p', 'code': 'unsupported_parameter'}}

#### 결과 해석

| 설정 | 서로 다른 응답 수 (5회 실행 예시) | 이유 |
|---|---|---|
| 보수적 (온도 0.1, `top_p` 0.3) | 1 / 5 | 확률이 가장 높은 이름만 골라 매번 같은 응답 |
| 창의적 (온도 1.5, `top_p` 0.95) | 5 / 5 | 확률이 낮은 이름도 골라 매번 다른 응답 |

- 효과는 한 번의 응답이 아니라 **반복 실행했을 때의 분포**에서 드러납니다.
- 일관된 답이 필요한 작업(코드, 분류, 추출)은 낮게, 다양한 답이 필요한 작업(이름, 아이디어)은 높게 설정합니다.
- 숫자는 `gpt-4o` 실행 예시이며, 실행할 때마다 달라질 수 있습니다.

In [18]:
# 여러 번 호출하므로 비용 부담이 적은 경량 모델을 사용합니다. (Groq: gpt-oss-20b / OpenAI: gpt-4o-mini)
# 제공자/모델은 위 공통 설정 셀의 get_llm 을 사용합니다. (max_retries=5 는 get_llm 에서 공통 적용)

# 같은 모델에서 temperature(와 페널티)만 다르게 비교합니다.
# top_p 는 temperature 와 같은 계열의 옵션이라, 함께 바꾸면 어떤 효과인지 구분하기 어려워 기본값(1.0)으로 둡니다.
COMMON = dict(
    light=True,                # 경량 모델 사용
    reasoning_effort="low",    # gpt-oss 는 추론 토큰도 max_tokens 에 포함 → 추론량을 줄여 본문이 비지 않게 함 (Groq 전용, OpenAI 선택 시 무시)
    max_tokens=300,            # 추론 토큰을 포함한 상한이므로 짧은 답이라도 여유 있게 설정
    provider="openai"
)

# 보수적 설정: 확률이 가장 높은 단어 위주로 선택 → 실행할 때마다 거의 같은 결과
llm_conservative = get_llm(temperature=0.0, **COMMON)

# 창의적 설정: 확률이 낮은 단어도 선택하고, 이미 쓴 표현·주제는 피하도록 유도 → 실행할 때마다 다른 결과
llm_creative = get_llm(temperature=1.2, presence_penalty=0.6, frequency_penalty=0.3, **COMMON)


def run_many(llm, question: str, n: int) -> list[str]:
    """같은 질문을 n번 실행해 응답 문자열 리스트를 반환합니다. (무료 한도를 고려해 동시 실행 수 제한)"""
    results = llm.batch([question] * n, config={"max_concurrency": 2})
    return [r.content.strip() for r in results]


def show(title: str, outputs: list[str]) -> None:
    """응답 목록과 '서로 다른 응답 수'를 출력합니다."""
    print(f"[{title}] 서로 다른 응답 수: {len(set(outputs))} / {len(outputs)}")
    for i, o in enumerate(outputs, 1):
        print(f"  {i}. {o}")
    print()


# 실험 1: 정답이 하나가 아닌 짧은 질문 (5회)
question1 = "신데렐라 이야기에 새로 등장할 마법사 캐릭터의 이름을 하나만 지어 주세요. 이름만 출력하세요."

print("=" * 70)
print(f" 실험 1) {question1}")
print("=" * 70)
show("보수적: temperature=0.0", run_many(llm_conservative, question1, 5))
show("창의적: temperature=1.2, presence=0.6, frequency=0.3", run_many(llm_creative, question1, 5))

# 실험 2: 새로운 배경으로 쓸 도시를 고르는 질문 (후보가 많고 답이 짧아 응답이 달라지는지 비교하기 좋음, 3회)
question2 = "신데렐라 이야기의 새로운 배경으로 어울리는 도시 하나를 골라 도시 이름만 출력하세요."

print("=" * 70)
print(f" 실험 2) {question2}")
print("=" * 70)
show("보수적: temperature=0.0", run_many(llm_conservative, question2, 3))
show("창의적: temperature=1.2, presence=0.6, frequency=0.3", run_many(llm_creative, question2, 3))

print("=" * 70)
print(" 기대 결과")
print("   • 보수적 설정: 같은 질문에 거의 같은 응답이 반복됨 (서로 다른 응답 수가 적음)")
print("   • 창의적 설정: 실행할 때마다 다른 이름·배경이 등장함 (서로 다른 응답 수가 많음)")
print("=" * 70)

[LLM 생성] provider=openai | model=gpt-4o-mini | temperature=0.0
[LLM 생성] provider=openai | model=gpt-4o-mini | temperature=1.2
 실험 1) 신데렐라 이야기에 새로 등장할 마법사 캐릭터의 이름을 하나만 지어 주세요. 이름만 출력하세요.
[보수적: temperature=0.0] 서로 다른 응답 수: 1 / 5
  1. 루미엘
  2. 루미엘
  3. 루미엘
  4. 루미엘
  5. 루미엘

[창의적: temperature=1.2, presence=0.6, frequency=0.3] 서로 다른 응답 수: 5 / 5
  1. 오르마디아
  2. 루미에르
  3. 루미엘
  4. 루미넬 마가란
  5. 루미나르

 실험 2) 신데렐라 이야기의 새로운 배경으로 어울리는 도시 하나를 골라 도시 이름만 출력하세요.
[보수적: temperature=0.0] 서로 다른 응답 수: 1 / 3
  1. 바르셀로나
  2. 바르셀로나
  3. 바르셀로나

[창의적: temperature=1.2, presence=0.6, frequency=0.3] 서로 다른 응답 수: 3 / 3
  1. 프라하
  2. 스위스의 취리히
  3. 비엔나

 기대 결과
   • 보수적 설정: 같은 질문에 거의 같은 응답이 반복됨 (서로 다른 응답 수가 적음)
   • 창의적 설정: 실행할 때마다 다른 이름·배경이 등장함 (서로 다른 응답 수가 많음)


#### 실험 결과 해석

| 실험 | 보수적 (온도 0.0) | 창의적 (온도 1.2) |
|---|---|---|
| 1. 마법사 이름 | 1 / 5 (매번 같은 이름) | 5 / 5 |
| 2. 배경 도시 | 1 / 3 (매번 같은 도시) | 3 / 3 |

- 낮은 온도는 응답이 고정되고, 높은 온도는 실행할 때마다 달라집니다. (`gpt-4o-mini` 실행 예시)
- `서로 다른 응답 수`는 문자열이 다른지만 세므로, 비슷한 이름(`루미-`)이나 같은 계열(유럽 도시)도 다른 응답으로 셉니다. 온도는 응답의 폭을 넓힐 뿐 모델의 큰 경향까지 바꾸지는 못합니다.

#### 모델에 추가적인 파라미터를 전달 (`bind`)
- `bind()`는 **이미 만든 모델의 파라미터를 덮어쓴 새 모델**을 만듭니다. (원본 모델은 바뀌지 않음)
- `max_tokens`: 50(기본) → 150으로 늘리면 더 긴 응답이 나옵니다.
- `stop=["."]`: 첫 번째 마침표에서 응답을 중단합니다.
- `temperature`: 기본값(1.0)과 비교하려고 0.0 / 1.2로 `bind`하고, 같은 질문을 **5회 반복**해 응답이 얼마나 달라지는지 봅니다.

In [24]:
from langchain_core.prompts import ChatPromptTemplate

# 프롬프트 설정 (천문학 질문에 대한 답변을 생성하는 시스템)
prompt = ChatPromptTemplate.from_messages([
    ("system", "이 시스템은 천문학 질문에 대해 명확하고 자세한 답변을 제공할 수 있습니다."),
    ("user", "{user_input}"),
])

# 기본 모델 설정: max_tokens=50 (temperature 는 모델 기본값, 반복 실행이 있어 경량 모델 사용)
base_model = get_llm(max_tokens=50, light=True, provider="openai")

# 질문 설정
long_question = "별이 어떻게 태어나고 죽는지 단계별로 자세히 설명해 주세요."    # max_tokens / stop 비교용 (긴 답변)
name_question = "새로 발견한 행성에 붙일 이름을 하나만 지어 주세요. 이름만 출력하세요."    # temperature 비교용 (짧은 답변)
N_RUNS = 5  # temperature 비교 반복 횟수

# 1) max_tokens: 기본(50) vs bind 로 150 으로 변경
print("\n [Before] 기본 max_tokens=50")
print((prompt | base_model).invoke({"user_input": long_question}).content)

print("\n" + "=" * 100 + "\n")

print(" [After] bind(max_tokens=150) → 더 긴 응답")
print((prompt | base_model.bind(max_tokens=150)).invoke({"user_input": long_question}).content)

print("\n" + "=" * 100 + "\n")

# 2) stop: 첫 번째 마침표에서 응답 중단 (max_tokens=50 은 그대로)
print(" [After] bind(stop=['.']) → 첫 번째 마침표에서 응답 중단")
print((prompt | base_model.bind(stop=["."])).invoke({"user_input": long_question}).content)

print("\n" + "=" * 100 + "\n")

# 3) temperature: 같은 질문을 N_RUNS 번 실행해 응답의 다양성 비교 (기본값은 1.0)
for temp in (0.0, 1.2):
    chain = prompt | base_model.bind(temperature=temp)
    answers = [r.content.strip() for r in chain.batch([{"user_input": name_question}] * N_RUNS, config={"max_concurrency": 3})]
    print(f" [After] bind(temperature={temp}) → 서로 다른 응답 수: {len(set(answers))} / {N_RUNS}")
    for i, answer in enumerate(answers, 1):
        print(f"  {i}. {answer}")
    print()

[LLM 생성] provider=openai | model=gpt-4o-mini | temperature=None

 [Before] 기본 max_tokens=50
별의 태어남과 죽음은 우주에서 일어나는 복잡한 과정으로, 여러 단계로 나눌 수 있습니다. 이 과정은 별의 질량에 따라 다소 차이가 있지만, 일반적인 단계를 아래


 [After] bind(max_tokens=150) → 더 긴 응답
별의 탄생과 죽음은 우주에서의 중요한 과정으로, 이들은 여러 단계를 거칩니다. 다음은 별의 생애를 단계별로 설명한 것입니다.

### 별의 탄생

1. **가스와 먼지 구름 (성간 물질)**:
   - 별은 거대한 가스와 먼지 구름에서 시작됩니다. 이 구름은 주로 수소와 헬륨으로 이루어져 있으며, 성간 물질이라고 불립니다.

2. **중력 수축**:
   - 특정한 자극(예: 슈퍼노바 폭발, 근처의 별의 중력 등)으로 인해, 구름의 일부가 불


 [After] bind(stop=['.']) → 첫 번째 마침표에서 응답 중단
별의 탄생과 죽음은 우주에서 일어나는 놀라운 과정으로, 여러 단계를 거칩니다


 [After] bind(temperature=0.0) → 서로 다른 응답 수: 1 / 5
  1. 아스트로니아
  2. 아스트로니아
  3. 아스트로니아
  4. 아스트로니아
  5. 아스트로니아

 [After] bind(temperature=1.2) → 서로 다른 응답 수: 5 / 5
  1. 헬리오스
  2. 엘레우시아
  3. 예레미아스
  4. 에코스톰
  5. 엑스플로라

